# Test Tanzanian ALPR

Run the trained plate detector + OCR on a photo. Change `IMAGE` in the next cell, or leave it to pick a labelled example.

Kernel: **tz-alpr** (`.venv`).

In [ ]:
from __future__ import annotations

import json
import os
import random
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

REPO = Path("..").resolve()
if not (REPO / "src" / "tz_alpr").exists():
    REPO = Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

os.environ.setdefault("TZ_ALPR_DEVICE", "cuda:0")
os.environ.setdefault("TZ_ALPR_RUNTIME", "torch")
os.environ.setdefault("TZ_ALPR_OCR_WEIGHTS", "models/ocr/v1/ocr_crnn.pt")
os.environ.setdefault("TZ_ALPR_PLATE_DETECTOR_WEIGHTS", "models/plate_detector/v1/plate_yolo.pt")

# Set to a path, or None to use the labelled T336CAG example.
IMAGE: Path | None = None

images_dir = REPO / "labeled_images"
if IMAGE is None:
    matches = sorted(images_dir.glob("T336CAG-*.jpg"))
    IMAGE = matches[0] if matches else next(images_dir.glob("*.jpg"))

print("repo   ", REPO)
print("image  ", IMAGE)

In [ ]:
from tz_alpr.pipeline import get_pipeline
from tz_alpr.utils.image_io import imread

pipeline = get_pipeline("configs/inference.yaml")
print("ocr    ", pipeline._ocr.backend)
print("plate  ", pipeline._plate_detector.name)
print("vehicle", pipeline._vehicle_detector.name if pipeline._vehicle_detector else None)

In [ ]:
def annotate(image_bgr: np.ndarray, response) -> np.ndarray:
    vis = image_bgr.copy()
    for r in response.results:
        x, y, w, h = r.plate_bbox.x, r.plate_bbox.y, r.plate_bbox.width, r.plate_bbox.height
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 220, 80), 3)
        label = f"{r.plate}  {r.confidence:.2f}  {r.review_status}"
        cv2.putText(vis, label, (x, max(24, y - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 220, 80), 2, cv2.LINE_AA)
        if r.vehicle.bbox is not None:
            b = r.vehicle.bbox
            cv2.rectangle(vis, (b.x, b.y), (b.x + b.width, b.y + b.height), (80, 160, 255), 2)
    return vis


def show(image_bgr: np.ndarray, title: str = "") -> None:
    plt.figure(figsize=(11, 8))
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()


image = imread(IMAGE)
response = pipeline.read_image(image)
print(json.dumps(response.model_dump(), indent=2))
show(annotate(image, response), f"{IMAGE.name}  {response.processing_time_ms} ms")

## Batch check against `labels.jsonl`

Sample `N` labelled photos and compare predicted plate vs ground truth.

In [ ]:
N = 12
SEED = 7

labels = []
with (REPO / "labels.jsonl").open() as fh:
    for line in fh:
        rec = json.loads(line)
        path = images_dir / rec["image"]
        if path.exists():
            labels.append((path, rec["plate_text"].strip().upper()))

rng = random.Random(SEED)
sample = rng.sample(labels, min(N, len(labels)))

rows = []
for path, truth in sample:
    img = imread(path)
    resp = pipeline.read_image(img)
    pred = resp.results[0].plate if resp.results else ""
    conf = resp.results[0].confidence if resp.results else 0.0
    ok = pred == truth
    rows.append((ok, path, truth, pred, conf, resp, img))
    print(f"{'OK' if ok else 'MISS':4}  {truth:10} -> {pred:10}  p={conf:.2f}  {path.name}")

hits = sum(1 for ok, *_ in rows if ok)
print(f"\nexact match  {hits}/{len(rows)}  ({hits / max(1, len(rows)):.0%})")

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 11))
for ax, (ok, path, truth, pred, conf, resp, img) in zip(axes.ravel(), rows):
    vis = annotate(img, resp)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{'OK' if ok else 'MISS'}  {truth} → {pred or '—'}", color=("#1a7f37" if ok else "#c0392b"))
    ax.axis("off")
plt.tight_layout()
plt.show()

## Optional: hit the running API

The live server is on port **8081** (8080 is Open WebUI).

In [ ]:
import httpx

API = "http://127.0.0.1:8081"
print(httpx.get(f"{API}/health", timeout=5).json())
print(httpx.get(f"{API}/version", timeout=5).json())

with IMAGE.open("rb") as fh:
    api_resp = httpx.post(f"{API}/v1/plate-reader", files={"upload": (IMAGE.name, fh, "image/jpeg")}, timeout=30)
print(json.dumps(api_resp.json(), indent=2))